# Notebook 01 — Data Collection & Enrichment

**Goal:** Load the Last.fm Dataset 1K baseline, fetch updated 5-year scrobble histories via the Last.fm API, and enrich with Spotify genre and audio feature data.

**Inputs:**
- `data/raw/userid-timestamp-artid-artname-traid-traname.tsv` (2.53 GB)
- `data/raw/userid-profile.tsv` (38 KB)
- Last.fm API credentials in `.env`
- Spotify API credentials in `.env`

**Outputs (all in `data/processed/`):**
- `scrobbles_baseline.parquet` — original 2009 dataset
- `scrobbles_updated.parquet` — merged baseline + last-5-year API data
- `profiles.parquet` — user demographics
- `artist_genres.parquet` — Spotify genre lookup table
- `audio_features.parquet` — Spotify audio features for top tracks

> **Note:** This notebook only needs to run once. All subsequent notebooks read from the saved Parquet files.

In [ ]:
import logging
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

## 1. Verify environment and credentials

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

required_vars = ['LASTFM_API_KEY', 'LASTFM_API_SECRET', 'SPOTIFY_CLIENT_ID', 'SPOTIFY_CLIENT_SECRET']
for var in required_vars:
    val = os.environ.get(var, '')
    status = '✓' if val else '✗ MISSING'
    print(f'  {var}: {status}')

## 2. Load baseline dataset (Last.fm Dataset 1K)

The large TSV is read in chunks to manage memory.

In [ ]:
from src.data.loader import load_scrobbles, load_profiles, load_config

cfg = load_config('../configs/config.yaml')

baseline_tsv = Path('../') / cfg['paths']['raw_scrobbles']
profiles_tsv = Path('../') / cfg['paths']['raw_profiles']

print(f'Baseline TSV exists: {baseline_tsv.exists()} ({baseline_tsv})')
print(f'Profiles TSV exists: {profiles_tsv.exists()} ({profiles_tsv})')

In [ ]:
# Load profiles (small — fast)
profiles = load_profiles(
    profiles_tsv,
    save_path='../data/processed/profiles.csv'
)
print(f'Profiles: {len(profiles)} users')
profiles.head()

In [ ]:
# Profile demographics summary
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

profiles['gender'].value_counts().plot(kind='bar', ax=axes[0], title='Gender Distribution')
profiles['age'].dropna().hist(ax=axes[1], bins=20, title='Age Distribution')
profiles['country'].value_counts().head(10).plot(kind='barh', ax=axes[2], title='Top 10 Countries')

plt.tight_layout()
plt.savefig('../outputs/figures/demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Load baseline scrobbles (takes a few minutes for the 2.53 GB file)
_baseline_csv = Path('../data/processed/scrobbles_baseline.csv')
if _baseline_csv.exists():
    baseline = pd.read_csv(_baseline_csv)
    print(f'Loaded from cache: {len(baseline):,} scrobbles, {baseline["userid"].nunique()} users')
    print(f'Date range: {baseline["timestamp"].min()} → {baseline["timestamp"].max()}')
elif baseline_tsv.exists():
    baseline = load_scrobbles(
        baseline_tsv,
        chunksize=500_000,
        save_parquet=str(_baseline_csv)
    )
    print(f'Baseline: {len(baseline):,} scrobbles, {baseline["userid"].nunique()} users')
    print(f'Date range: {baseline["timestamp"].min()} → {baseline["timestamp"].max()}')
else:
    print('Baseline TSV not found — will use API data only.')
    baseline = None
baseline.head() if baseline is not None else None

## 3. Fetch updated scrobbles via Last.fm API

This pulls the last 5 years of listening history for all users still active on Last.fm.
Results are saved incrementally per user so the process can be resumed if interrupted.

In [ ]:
_updated_csv = Path('../data/processed/scrobbles_updated.csv')
if _updated_csv.exists():
    scrobbles = pd.read_csv(_updated_csv)
    network = None
    print(f'Loaded merged scrobbles from cache: {len(scrobbles):,} rows, {scrobbles["userid"].nunique()} users')
else:
    from src.data.lastfm_client import build_network, fetch_all_users
    network = build_network()
    print('Last.fm network initialised.')
    userids = profiles['userid'].dropna().unique().tolist()
    print(f'Fetching updated scrobbles for {len(userids)} users...')

In [ ]:
# This cell makes real API calls — skipped automatically if cache exists.
if network is not None:
    dc = cfg['data_collection']
    updated = fetch_all_users(
        network=network,
        userids=userids,
        lookback_years=dc['lookback_years'],
        page_size=dc['lastfm_page_size'],
        request_delay=dc['lastfm_request_delay'],
        min_scrobbles=dc['min_scrobbles_threshold'],
        save_dir='../data/processed/lastfm_user_cache',
        resume=True,
    )
    print(f'Updated scrobbles: {len(updated):,} rows, {updated["userid"].nunique()} active users')

In [ ]:
# Merge baseline + updated, deduplicate
import pandas as pd

if not _updated_csv.exists():
    frames = [f for f in [baseline, updated] if f is not None and len(f) > 0]
    scrobbles = pd.concat(frames, ignore_index=True)
    scrobbles = scrobbles.drop_duplicates(
        subset=['userid', 'timestamp', 'artist_name', 'track_name']
    ).sort_values(['userid', 'timestamp']).reset_index(drop=True)
    scrobbles.to_csv(str(_updated_csv), index=False)
    print(f'Merged & saved: {len(scrobbles):,} scrobbles, {scrobbles["userid"].nunique()} users')
else:
    print(f'Using cached: {len(scrobbles):,} scrobbles, {scrobbles["userid"].nunique()} users')

## 4. Enrich with Spotify genres

In [ ]:
_genres_csv = Path('../data/processed/artist_genres.csv')
if not _genres_csv.exists():
    from src.data.spotify_client import build_client, fetch_artist_genres
    sp = build_client()
    unique_artists = scrobbles['artist_name'].dropna().unique().tolist()
    print(f'Unique artists to look up: {len(unique_artists):,}')
else:
    sp = None
    print(f'Artist genres already cached ({_genres_parquet})')

In [ ]:
if _genres_csv.exists():
    artist_genres = pd.read_csv(_genres_csv)
    print(f'Loaded from cache: {len(artist_genres):,} artists')
else:
    # Fetches genres with persistent cache — safe to re-run
    artist_genres = fetch_artist_genres(
        sp,
        unique_artists,
        cache_path='../data/processed/spotify_artist_cache.csv',
        request_delay=0.1,
    )
    artist_genres.to_csv(str(_genres_csv), index=False)

found = artist_genres['spotify_artist_id'].notna().sum()
print(f'Artists matched on Spotify: {found:,} / {len(artist_genres):,}')

## 5. Enrich with Spotify audio features

In [ ]:
_audio_csv = Path('../data/processed/audio_features.csv')
if not _audio_csv.exists():
    from src.data.spotify_client import fetch_audio_features
    if sp is None:
        from src.data.spotify_client import build_client
        sp = build_client()

    # Sample top 20 tracks per user to keep API calls manageable
    TOP_TRACKS_PER_USER = 20
    track_sample = (
        scrobbles.groupby(['userid', 'artist_name', 'track_name'])
        .size().reset_index(name='play_count')
        .sort_values(['userid', 'play_count'], ascending=[True, False])
        .groupby('userid').head(TOP_TRACKS_PER_USER)
    )
    track_pairs = list(set(zip(track_sample['artist_name'], track_sample['track_name'])))
    print(f'Unique (artist, track) pairs to enrich: {len(track_pairs):,}')
else:
    print(f'Audio features already cached ({_audio_parquet})')

In [ ]:
if _audio_csv.exists():
    audio_features = pd.read_csv(_audio_csv)
    print(f'Loaded from cache: {len(audio_features):,} tracks')
else:
    audio_features = fetch_audio_features(
        sp,
        track_pairs,
        cache_path='../data/processed/spotify_audio_features_cache.csv',
        request_delay=0.1,
    )
    audio_features.to_csv(str(_audio_csv), index=False)
    print(f'Audio features fetched: {len(audio_features):,} tracks')
audio_features.head()

## Summary

All enriched data is saved to `data/processed/`. Proceed to **Notebook 02** for feature engineering.